# Phase 7: Explainability Layer (Subset Generator)

Notebook ini khusus menghasilkan *structured explanation* (JSON) untuk sebagian kecil dataset (5 sampel) pada kondisi AB dan BA.
Data ini akan di-*cache* untuk diumpankan ke Streamlit Demo dalam Mode Replay.

In [1]:
!pip install -q transformers accelerate bitsandbytes datasets pyarrow pandas

In [2]:
import os
import json

# Konfigurasi MLOps Arsitektur
PROJECT_ROOT: str = '/content/drive/MyDrive/LLMReliability'
DATA_PROCESSED_DIR: str = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR: str = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR: str = os.path.join(PROJECT_ROOT, 'results')

# Mengarahkan Cache HuggingFace ke Google Drive agar Reusable (Sesuai AGENTS.md)
os.environ['HF_HOME'] = MODELS_DIR

# Mount Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Authenticate to HuggingFace
    secrets_path = '/content/drive/MyDrive/secrets.json'
    if os.path.exists(secrets_path):
        with open(secrets_path, 'r') as f:
            secrets = json.load(f)
        hf_token = secrets.get("HF_TOKEN")
        if hf_token:
            from huggingface_hub import login
            login(token=hf_token)
            print("[INFO] Successfully authenticated to HuggingFace Hub.")
except ImportError:
    print("[INFO] Not in Colab environment. Using local storage.")

for d in [DATA_PROCESSED_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"[INFO] Storage Architecture initialized at {PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Successfully authenticated to HuggingFace Hub.
[INFO] Storage Architecture initialized at /content/drive/MyDrive/LLMReliability


In [3]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

os.chdir('/content/drive/MyDrive/LLMReliability')

# Ambil 5 sampel dari main dataset
print("[INFO] Memuat dataset...")
df = pd.read_parquet('data/processed/indopref_main.parquet')
subset_df = df.head(5).copy() # Ambil 5 record pertama

[INFO] Memuat dataset...


In [4]:
print("[INFO] Inisialisasi Qwen3-4B...")
model_id = "Qwen/Qwen3-4B-Instruct-2507"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("[INFO] Model successfully loaded into GPU memory!")

[INFO] Inisialisasi Qwen3-4B...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[INFO] Model successfully loaded into GPU memory!


In [5]:
def format_json_prompt(user_input, resp_A, resp_B):
    base = "Bandingkan Response A dan Response B. Pilih respons yang lebih baik."
    prompt = f"""{base}

Anda HARUS mengevaluasi berdasarkan tiga kriteria: Accuracy/Correctness, Relevance, dan Clarity.
Keluarkan hasil evaluasi Anda HANYA dalam format JSON valid dengan struktur persis seperti berikut, tanpa tambahan teks atau markdown backticks:
{{
  "decision": "A",
  "criteria": {{
    "accuracy": {{ "A": 9, "B": 8 }},
    "relevance": {{ "A": 9, "B": 8 }},
    "clarity": {{ "A": 9, "B": 8 }}
  }},
  "explanation": {{
    "accuracy": "Penjelasan singkat",
    "relevance": "Penjelasan singkat",
    "clarity": "Penjelasan singkat"
  }},
  "evidence": [
    {{
      "criterion": "kriteria",
      "candidate": "A",
      "evidence": "kutipan teks respons",
      "assessment": "supported atau unsupported"
    }}
  ]
}}

[INPUT CONTEXT]
Input: {user_input}

Response A:
{resp_A}

Response B:
{resp_B}
"""
    return prompt

def run_inference(user_input, resp_A, resp_B):
    raw_prompt = format_json_prompt(user_input, resp_A, resp_B)
    messages = [
        {"role": "system", "content": "You are a rigid AI evaluator. You follow instructions perfectly. Always output valid raw JSON without code blocks or markdown."},
        {"role": "user", "content": raw_prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        
    generated_tokens = outputs[0][input_len:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

In [6]:
results = []
print("[INFO] Menjalankan inferensi untuk 5 subset...")

for index, row in subset_df.iterrows():
    print(f"Processing sample ID {row['original_id']}...")
    
    # --- Posisi AB ---
    raw_ab = run_inference(row['input'], row['response_1'], row['response_2'])
    
    # --- Posisi BA ---
    raw_ba = run_inference(row['input'], row['response_2'], row['response_1'])
    
    results.append({
        "sample_id": row["original_id"],
        "prompt": row["input"],
        "cand_1": row["response_1"],
        "cand_2": row["response_2"],
        "human_preference": row.get("chosen_model", "Unknown"),
        "json_output_ab": raw_ab,
        "json_output_ba": raw_ba
    })

out_path = 'data/processed/explainability_subset.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for res in results:
        f.write(json.dumps(res, ensure_ascii=False) + '\n')

print(f"\n[SUCCESS] Subset explainability disimpan ke {out_path}!")

[INFO] Menjalankan inferensi untuk 5 subset...
Processing sample ID 238...
Processing sample ID 327...
Processing sample ID 423...
Processing sample ID 403...
Processing sample ID 454...

[SUCCESS] Subset explainability disimpan ke data/processed/explainability_subset.jsonl!
